# Step 01 — NAIP Imagery Tile Service URL Generator

This notebook uses **Google Earth Engine** to mosaic [NAIP](https://naip-usdaonline.hub.arcgis.com/) imagery for each available year over a bounding-box Area of Interest (AOI), then generates XYZ tile service URLs (true-color RGB and false-color infrared) and saves them to a CSV.

### Efficiency Note
The geometry provided below creates a bounding box that limits the mosaic extent, significantly improving speed and resource usage compared to processing wider areas.

### Libraries Used
- [earthengine-api](https://developers.google.com/earth-engine/guides/python_install) — Google Earth Engine Python client
- [geemap](https://geemap.org/) — Interactive GEE mapping utilities

## 1. Install Necessary Libraries

Install the Earth Engine API and [geemap](https://geemap.org/) helper package.

In [1]:
!pip install -q --upgrade earthengine-api geemap

## 2. Initialize Google Earth Engine

Authenticate and initialize the Earth Engine session. If you have not previously authenticated, a browser window will open for you to sign in.

In [2]:
import ee
import json

# Initialize the Earth Engine module
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

## 3. Define the Area of Interest (AOI)

Define the bounding box as inline GeoJSON and convert it to an Earth Engine geometry. This limits the NAIP mosaic to the study area.

> **Tip:** Replace the coordinates below with your own AOI, or load from a file with `json.load()`. The geometry should match the AOI used in **Step 02** and **Step 03**.

In [3]:
# Define the GeoJSON Area of Interest (bounding box)
roi_json = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "name": "mortalitree_czu_download_bbox"
      },
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [-122.346241721068, 37.0180037353093],
            [-122.087073088937, 37.0180037353093],
            [-122.087073088937, 37.2720781887031],
            [-122.346241721068, 37.2720781887031],
            [-122.346241721068, 37.0180037353093]
          ]
        ]
      }
    }
  ]
}

# Convert the GeoJSON object to an Earth Engine Geometry
geometry = ee.FeatureCollection(roi_json).geometry()

## 4. Query NAIP Imagery Collection

Filter the [USDA NAIP](https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ) image collection by the AOI and a broad date range, then extract the distinct years of available imagery.

In [4]:
# Filter the NAIP collection by AOI and date range
dataset = ee.ImageCollection('USDA/NAIP/DOQQ') \
    .filterBounds(geometry) \
    .filterDate('2019-01-01', '2025-12-31')

# Extract distinct years of available imagery
years = dataset.aggregate_array('system:time_start') \
    .map(lambda time: ee.Date(time).get('year')) \
    .distinct() \
    .sort()

year_list = years.getInfo()
print(f"Found NAIP imagery for years: {year_list}")

Found NAIP imagery for years: [2020, 2022]


## 5. Generate Tile Service URLs and Export to CSV

For each NAIP year, create a mosaic clipped to the AOI, compute min/max statistics for visualization, and generate two XYZ tile service URLs:
- **RGB (True Color)** — Bands R, G, B
- **IR (False Color Infrared)** — Bands N, R, G

The output CSV (`../output/tile_urls.csv`) has columns `name` and `tile_url`, and is consumed by **Step 02** (tile downloading) and **Step 03** (tile grid preview).

In [5]:
import os
import csv

# Define output file path (consumed by Step 02 and Step 03)
output_file = '../output/tile_urls.csv'

# Ensure output directory exists
os.makedirs(os.path.dirname(output_file), exist_ok=True)

print(f"Processing {len(year_list)} years and saving to {output_file}...\n")

with open(output_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["name", "tile_url"])

    for year in year_list:
        yearly_col = dataset.filterDate(
            ee.Date.fromYMD(year, 1, 1),
            ee.Date.fromYMD(year, 12, 31)
        )

        # Create mosaic and clip to AOI geometry
        mosaic = yearly_col.mosaic().clip(geometry)

        # Check available bands for this year
        try:
            band_names = mosaic.bandNames().getInfo()
        except Exception as e:
            continue

        # Calculate min/max statistics for visualization stretch
        stats = mosaic.reduceRegion(
            reducer=ee.Reducer.minMax(),
            geometry=geometry,
            scale=10,
            bestEffort=True,
            maxPixels=1e9
        )
        val = stats.getInfo()

        # --- RGB (True Color) ---
        if set(['R', 'G', 'B']).issubset(band_names):
            viz_rgb = {
                'bands': ['R', 'G', 'B'],
                'min': [val['R_min'], val['G_min'], val['B_min']],
                'max': [val['R_max'], val['G_max'], val['B_max']]
            }
            map_id = mosaic.getMapId(viz_rgb)
            url_format = map_id["tile_fetcher"].url_format
            name = f"gee_naip_{year}"
            writer.writerow([name, url_format])
            print(f"  {name}")

        # --- IR (False Color Infrared) ---
        if 'N' in band_names and set(['R', 'G']).issubset(band_names):
            viz_ir = {
                'bands': ['N', 'R', 'G'],
                'min': [val['N_min'], val['R_min'], val['G_min']],
                'max': [val['N_max'], val['R_max'], val['G_max']]
            }
            map_id_ir = mosaic.getMapId(viz_ir)
            url_format_ir = map_id_ir["tile_fetcher"].url_format
            name_ir = f"gee_naip_ir_{year}"
            writer.writerow([name_ir, url_format_ir])
            print(f"  {name_ir}")

print(f"\nSaved {output_file}")

Processing 2 years and saving to ../output/tile_urls.csv...

  gee_naip_2020
  gee_naip_ir_2020
  gee_naip_2022
  gee_naip_ir_2022

Saved ../output/tile_urls.csv
